# Mirror Channels (Self-Contained)
No imports from `mirror.py` or `open_ai_client.py`.

In [1]:
import copy
import csv
import json
import os
import sys
import time
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Set
from urllib.parse import urlparse

from dotenv import load_dotenv
from google.cloud import bigquery
from google.cloud import firestore
from openai import OpenAI

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "src").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

env_path = project_root / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print(f"Loaded env: {env_path}")
else:
    load_dotenv()
    print("Loaded env via default dotenv lookup")

SUMMARY_INSTRUCTIONS = """
Using the metadata above and any reliable public knowledge, write a structured JSON object describing this YouTube channel.
Use this template exactly, filling in all fields where possible:
{
  \"overview\": \"...\",
  \"content_topics\": [\"...\"],
  \"signature_series\": [\"...\"],
  \"target_audience\": \"...\",
  \"tone_style\": \"...\",
  \"keywords\": [\"...\"],
  \"video_success_factors\": [\"...\"],
  \"value_proposition\": \"...\"
}
Important:
- Exclude any content that is adult, violent, spammy, misleading, or inappropriate for general audiences.
- Only include topics, keywords, or series that are verifiable or clearly relevant to the channel.
- Keep all text factual and suitable for semantic search.
"""

AI_TEMPLATE = {
    "overview": None,
    "content_topics": [],
    "signature_series": [],
    "target_audience": None,
    "tone_style": None,
    "keywords": [],
    "video_success_factors": [],
    "value_proposition": None,
}

current_creds = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
if not current_creds or not os.path.exists(current_creds):
    fallback_service_json = project_root / "src" / "dump" / "service.json"
    if fallback_service_json.exists():
        os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(fallback_service_json)
        print(f"Using fallback credentials: {fallback_service_json}")

print("project_root:", project_root)


Loaded env via default dotenv lookup
project_root: E:\Git\lds\backend\practice\py-script


In [2]:
class ChannelInsightsClient:
    def __init__(
        self,
        chat_model: str = "gpt-4.1",
        embeddings_model: str = "text-embedding-3-large",
        max_retries: int = 3,
        retry_delay: float = 2.0,
        template: Optional[Dict[str, Any]] = None,
    ) -> None:
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("OPENAI_API_KEY is not set. Check your .env file.")
        self.client = OpenAI(api_key=api_key)
        self.chat_model = chat_model
        self.embeddings_model = embeddings_model
        self.max_retries = max_retries
        self.retry_delay = retry_delay
        self.template = template or {}

    def _extract_text(self, response: Any) -> str:
        text = getattr(response, "output_text", None)
        if text:
            return text.strip()

        text_output = ""
        if hasattr(response, "choices") and len(response.choices) > 0:
            if hasattr(response.choices[0], "message"):
                text_output = response.choices[0].message.content or ""

        if not text_output:
            for item in getattr(response, "output", []):
                for content in item.get("content", []):
                    text_output += content.get("text", "")

        return text_output.strip() or "No text found in response."

    def describe_channel(self, channel_name: str, context: Optional[str] = None, structured: bool = True) -> str:
        _ = structured
        system_prompt = "You are a helpful assistant that summarizes information about YouTube channels."
        template_str = json.dumps(self.template, indent=4) if self.template else "{}"
        user_prompt = (
            f"Create a structured JSON description for the YouTube channel '{channel_name}'. "
            f"Use the following JSON structure exactly, filling in all fields with factual information:\n\n{template_str} "
            "Exclude any adult, violent, spammy, or inappropriate content."
        )
        if context:
            user_prompt += f"\n\nRelevant context: {context}"

        messages: List[Dict[str, str]] = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]

        last_error: Optional[Exception] = None
        for attempt in range(self.max_retries):
            try:
                if hasattr(self.client, "responses"):
                    response = self.client.responses.create(
                        model=self.chat_model,
                        input=messages,
                        temperature=0.4,
                    )
                else:
                    response = self.client.chat.completions.create(
                        model=self.chat_model,
                        messages=messages,
                        temperature=0.4,
                    )
                summary = self._extract_text(response)
                if summary:
                    return " ".join(summary.split())
            except Exception as exc:
                last_error = exc
                time.sleep(self.retry_delay * (attempt + 1))

        raise RuntimeError(f"Failed to describe channel after {self.max_retries} retries: {last_error}")

    def describe_channel_json(self, channel_name: str, context: Optional[str] = None) -> str:
        raw_text = self.describe_channel(channel_name, context=context, structured=True)
        try:
            json_start = raw_text.find("{")
            json_end = raw_text.rfind("}") + 1
            if json_start != -1 and json_end != 0:
                json_part = raw_text[json_start:json_end]
                json.loads(json_part)
                return json_part
            json.loads(raw_text)
            return raw_text
        except json.JSONDecodeError:
            fallback = self.template.copy() if self.template else {"overview": raw_text}
            fallback["overview"] = raw_text
            return json.dumps(fallback, ensure_ascii=False, indent=4)

    def embed_text(self, text: str) -> List[float]:
        cleaned = text.strip()
        if not cleaned:
            raise ValueError("Text must be non-empty for embedding generation.")
        response = self.client.embeddings.create(model=self.embeddings_model, input=cleaned)
        return response.data[0].embedding

    def close(self) -> None:
        try:
            self.client.close()
        except Exception:
            pass

    def __del__(self) -> None:
        self.close()


class ChannelSemanticClient:
    def __init__(self, insights_client: Optional[ChannelInsightsClient] = None):
        self.insights_client = insights_client or ChannelInsightsClient()

    def generate_channel_embedding(self, channel_payload: Dict[str, Any]) -> Dict[str, Any]:
        meta = channel_payload.get("channel_metadata", {})
        structured = channel_payload.get("channel_description_structured", {})
        semantic_parts = [
            f"Title: {meta.get('title', '')}",
            f"Description: {meta.get('description', '')}",
            f"Topics: {', '.join(structured.get('content_topics', []))}",
            f"Value Proposition: {structured.get('value_proposition', '')}",
            f"Tone: {structured.get('tone_style', '')}",
        ]
        channel_payload["semantic_text"] = " ".join(semantic_parts)
        return channel_payload

    def close(self):
        self.insights_client.close()


In [3]:
def log(level: str, msg: str, report: List[Dict]) -> None:
    ts = datetime.utcnow().isoformat()
    print(f"[{ts}] {level} {msg}")
    report.append({"ts": ts, "level": level, "message": msg})


def bq_rows_are_equal(
    bq_client: bigquery.Client,
    table_id: str,
    id_field: str,
    parent_id: str,
    new_channel_id: str,
    report: List[Dict],
    row_num: int,
    name_field: Optional[str] = None,
    new_channel_name: Optional[str] = None,
) -> bool:
    if name_field:
        parent_norm_sql = f"""
            SELECT TO_JSON_STRING(t) AS row_json
            FROM (
                SELECT * REPLACE(@new_channel_id AS {id_field}, @new_channel_name AS {name_field})
                FROM `{table_id}`
                WHERE {id_field} = @parent_id
            ) AS t
        """
        params = [
            bigquery.ScalarQueryParameter("parent_id", "STRING", parent_id),
            bigquery.ScalarQueryParameter("new_channel_id", "STRING", new_channel_id),
            bigquery.ScalarQueryParameter("new_channel_name", "STRING", new_channel_name or ""),
        ]
    else:
        parent_norm_sql = f"""
            SELECT TO_JSON_STRING(t) AS row_json
            FROM (
                SELECT * REPLACE(@new_channel_id AS {id_field})
                FROM `{table_id}`
                WHERE {id_field} = @parent_id
            ) AS t
        """
        params = [
            bigquery.ScalarQueryParameter("parent_id", "STRING", parent_id),
            bigquery.ScalarQueryParameter("new_channel_id", "STRING", new_channel_id),
        ]

    compare_sql = f"""
    WITH parent_norm AS (
        {parent_norm_sql}
    ),
    target_rows AS (
        SELECT TO_JSON_STRING(t) AS row_json
        FROM (
            SELECT * FROM `{table_id}` WHERE {id_field} = @new_channel_id
        ) AS t
    )
    SELECT
      (SELECT COUNT(*) FROM (
          SELECT row_json FROM parent_norm
          EXCEPT DISTINCT
          SELECT row_json FROM target_rows
      )) AS missing_in_target,
      (SELECT COUNT(*) FROM (
          SELECT row_json FROM target_rows
          EXCEPT DISTINCT
          SELECT row_json FROM parent_norm
      )) AS extra_in_target
    """

    try:
        result = list(
            bq_client.query(compare_sql, job_config=bigquery.QueryJobConfig(query_parameters=params)).result(timeout=120)
        )[0]
        same = result["missing_in_target"] == 0 and result["extra_in_target"] == 0
        if same:
            log("[INFO]", f"Row {row_num}: BQ '{table_id.split('.')[-1]}' already up to date; skipping", report)
        return same
    except Exception as exc:
        log("[WARN]", f"Row {row_num}: BQ comparison failed ({exc}) - assuming diff", report)
        return False


def sync_bq_table_server_side(
    bq_client: bigquery.Client,
    table_id: str,
    id_field: str,
    parent_id: str,
    new_channel_id: str,
    report: List[Dict],
    row_num: int,
    name_field: Optional[str] = None,
    new_channel_name: Optional[str] = None,
) -> None:
    if name_field:
        insert_select = f"""
            SELECT * REPLACE(@new_channel_id AS {id_field}, @new_channel_name AS {name_field})
            FROM `{table_id}`
            WHERE {id_field} = @parent_id
        """
        params = [
            bigquery.ScalarQueryParameter("parent_id", "STRING", parent_id),
            bigquery.ScalarQueryParameter("new_channel_id", "STRING", new_channel_id),
            bigquery.ScalarQueryParameter("new_channel_name", "STRING", new_channel_name or ""),
        ]
    else:
        insert_select = f"""
            SELECT * REPLACE(@new_channel_id AS {id_field})
            FROM `{table_id}`
            WHERE {id_field} = @parent_id
        """
        params = [
            bigquery.ScalarQueryParameter("parent_id", "STRING", parent_id),
            bigquery.ScalarQueryParameter("new_channel_id", "STRING", new_channel_id),
        ]

    sql = f"""
    BEGIN TRANSACTION;
    DELETE FROM `{table_id}` WHERE {id_field} = @new_channel_id;
    INSERT INTO `{table_id}` {insert_select};
    COMMIT TRANSACTION;
    """

    try:
        bq_client.query(sql, job_config=bigquery.QueryJobConfig(query_parameters=params)).result(timeout=180)
    except Exception as sync_err:
        err_msg = str(sync_err)
        if "streaming buffer" in err_msg.lower() or "400" in err_msg:
            log("[WARN]", f"Row {row_num}: BQ sync deferred for '{table_id.split('.')[-1]}' (streaming buffer)", report)
            return
        raise sync_err

    count_sql = f"SELECT COUNT(*) AS cnt FROM `{table_id}` WHERE {id_field} = @new_channel_id"
    count_result = list(
        bq_client.query(
            count_sql,
            job_config=bigquery.QueryJobConfig(
                query_parameters=[bigquery.ScalarQueryParameter("new_channel_id", "STRING", new_channel_id)]
            ),
        ).result(timeout=60)
    )[0]
    log("[OK]", f"Row {row_num}: synced {count_result['cnt']} rows -> {table_id.split('.')[-1]}", report)


def build_openai_context(metadata: Dict[str, Any], new_channel_name: str) -> str:
    lines: List[str] = [
        f"You are describing a NEW YouTube channel called '{new_channel_name}'.",
        "The metadata below is from the parent channel and is only reference context.",
        "Tailor the description to the new channel identity.",
        "",
        "Parent channel metadata from YouTube Data API:",
    ]

    def add_line(label: str, value: Optional[Any]) -> None:
        if value is None:
            return
        if isinstance(value, list):
            if value:
                lines.append(f"{label}: {', '.join(str(i) for i in value)}")
        else:
            text = str(value).strip()
            if text:
                lines.append(f"{label}: {text}")

    add_line("Channel ID", metadata.get("channel_id"))
    add_line("Title", metadata.get("title"))
    add_line("Subtitle", metadata.get("subtitle"))
    add_line("Description", metadata.get("description"))
    add_line("Channel URL", metadata.get("channel_url"))
    add_line("Custom URL", metadata.get("custom_url"))
    add_line("Default Language", metadata.get("default_language"))
    add_line("Country", metadata.get("country"))
    add_line("Published At", metadata.get("published_at"))
    add_line("Keywords", metadata.get("tags"))
    add_line("Topic Categories", metadata.get("topic_categories"))

    return "\n".join(lines)


def validate_env() -> None:
    fs_project = os.getenv("FIRESTORE_PROJECT_ID", "")
    bq_project = os.getenv("GOOGLE_PROJECT_ID", "")
    if fs_project and bq_project:
        if ("prod" in fs_project.lower()) != ("prod" in bq_project.lower()):
            raise ValueError(
                f"Environment mismatch - Firestore uses '{fs_project}' but BigQuery uses '{bq_project}'."
            )


def derive_custom_url(url_or_handle: str) -> Optional[str]:
    if not url_or_handle:
        return None
    raw = url_or_handle.strip()
    if not raw:
        return None
    if raw.startswith("@"):
        return raw
    if raw.startswith("http://") or raw.startswith("https://"):
        parsed = urlparse(raw)
        path = (parsed.path or "").strip("/")
        return path or None
    return raw.strip("/")


In [4]:
def mirror_channels(csv_file: str, dry_run: bool = False) -> None:
    validate_env()

    google_project_id = os.getenv("GOOGLE_PROJECT_ID")
    firestore_database = os.getenv("FIRESTORE_DATABASE")

    if google_project_id is None or firestore_database is None:
        raise ValueError("Missing GCP_PROJECT_ID or FIRESTORE_DATABASE in env")

    db = firestore.Client(project=google_project_id, database=firestore_database)

    bq_project = google_project_id
    bq_dataset = os.getenv("BQ_DATASET", "krowten")
    bq_client = bigquery.Client(project=bq_project)

    source_col = db.collection("channels")
    target_col = db.collection("channels")

    report: List[Dict] = []
    insights_client: Optional[ChannelInsightsClient] = None
    semantic_client: Optional[ChannelSemanticClient] = None

    try:
        insights_client = ChannelInsightsClient(template=AI_TEMPLATE)
        semantic_client = ChannelSemanticClient(insights_client=insights_client)

        seen_targets: Set[str] = set()
        count = 0

        with open(csv_file, encoding="utf-8") as fh:
            reader = csv.DictReader(fh)

            for row_num, row in enumerate(reader, start=1):
                parent_id = row.get("Parent Channel ID", "").strip()
                new_channel_id = row.get("Channel ID", "").strip()
                new_channel_name = row.get("New Channel", "").strip()
                new_channel_url = row.get("Copycat URL 1", "").strip() or row.get("Copycat URL 2", "").strip()

                if not parent_id or not new_channel_id:
                    log("[WARN]", f"Row {row_num}: missing IDs; skipped", report)
                    continue
                if not new_channel_name:
                    log("[WARN]", f"Row {row_num}: missing channel name; skipped", report)
                    continue
                if new_channel_id in seen_targets:
                    log("[ERROR]", f"Row {row_num}: duplicate target '{new_channel_id}' in CSV; aborting", report)
                    raise ValueError(f"Duplicate target channel ID in CSV: {new_channel_id}")
                seen_targets.add(new_channel_id)

                parent_doc = source_col.document(parent_id).get()
                if not parent_doc.exists:
                    log("[ERROR]", f"Row {row_num}: parent '{parent_id}' not found; skipped", report)
                    continue

                original_data = parent_doc.to_dict()
                channel_metadata = copy.deepcopy(original_data.get("channel_metadata", {}))
                channel_metadata["title"] = new_channel_name
                if new_channel_url:
                    channel_metadata["channel_url"] = new_channel_url
                    derived_custom = derive_custom_url(new_channel_url)
                    if derived_custom:
                        channel_metadata["custom_url"] = derived_custom

                target_ref = target_col.document(new_channel_id)
                target_doc = target_ref.get()
                target_existed = target_doc.exists
                target_snapshot = target_doc.to_dict() if target_existed else None

                flags = copy.deepcopy(original_data.get("flags", {}))
                flags["is-external"] = False
                flags["is-mirrored"] = True

                log("[AI]", f"Row {row_num}: generating AI enrichment for '{new_channel_name}'...", report)
                ai_status = "success"
                try:
                    context = f"{build_openai_context(channel_metadata, new_channel_name)}\n{SUMMARY_INSTRUCTIONS}"
                    response_json = insights_client.describe_channel_json(new_channel_name, context=context)
                    structured_description = json.loads(response_json)

                    enrichment_payload = {
                        "channel_id": new_channel_id,
                        "channel_name": new_channel_name,
                        "channel_url": new_channel_url or channel_metadata.get("channel_url"),
                        "channel_description": structured_description.get("overview", channel_metadata.get("description")),
                        "channel_description_structured": structured_description,
                        "channel_metadata": channel_metadata,
                        "openai_raw_response": response_json,
                    }
                    metadata_gpt_enrichment = semantic_client.generate_channel_embedding(enrichment_payload)
                except Exception as ai_exc:
                    ai_status = "fallback"
                    log("[WARN]", f"Row {row_num}: AI enrichment failed ({ai_exc}). Using parent enrichment as fallback.", report)
                    metadata_gpt_enrichment = copy.deepcopy(original_data.get("metadata-gpt-enrichment", {}))

                mirrored_data = {
                    "flags": flags,
                    "matching-criteria": copy.deepcopy(original_data.get("matching-criteria", {})),
                    "metadata-gpt-enrichment": metadata_gpt_enrichment,
                    "metadata-keywords": copy.deepcopy(original_data.get("metadata-keywords", {})),
                    "name": new_channel_name,
                    "schedule": copy.deepcopy(original_data.get("schedule", {})),
                    "services": copy.deepcopy(original_data.get("services", {})),
                    "sidekick": {},
                    "client": original_data.get("client", "NETWORK"),
                    "parent-channel-id": parent_id,
                }

                if dry_run:
                    action = "UPDATE" if target_existed else "CREATE"
                    log("[DRY-RUN]", f"Would {action}: {new_channel_id} (AI: {ai_status})", report)
                    count += 1
                    continue

                try:
                    # target_ref.set(mirrored_data, merge=True)
                    action = "updated" if target_existed else "created"
                    log("[FS]", f"Row {row_num}: Firestore {action}: {parent_id} -> {new_channel_id}", report)
                except Exception as fs_err:
                    log("[ERROR]", f"Row {row_num}: Firestore write failed ({fs_err}); skipped", report)
                    continue

                tables = [
                    {"name": "channel_embeddings", "id_field": "channel_id", "name_field": "channel_name"},
                    {"name": "channel_video_embeddings", "id_field": "channel_id", "name_field": None},
                ]

                bq_success = True
                try:
                    for t in tables:
                        table_id = f"{bq_project}.{bq_dataset}.{t['name']}"
                        is_same = bq_rows_are_equal(
                            bq_client=bq_client,
                            table_id=table_id,
                            id_field=t["id_field"],
                            parent_id=parent_id,
                            new_channel_id=new_channel_id,
                            report=report,
                            row_num=row_num,
                            name_field=t["name_field"],
                            new_channel_name=new_channel_name,
                        )
                        if is_same:
                            continue
                        log("[BQ]", f"Row {row_num}: syncing BQ table '{t['name']}'", report)
                        # sync_bq_table_server_side(
                        #     bq_client=bq_client,
                        #     table_id=table_id,
                        #     id_field=t["id_field"],
                        #     parent_id=parent_id,
                        #     new_channel_id=new_channel_id,
                        #     report=report,
                        #     row_num=row_num,
                        #     name_field=t["name_field"],
                        #     new_channel_name=new_channel_name,
                        # )
                except Exception as bq_err:
                    bq_success = False
                    log("[ERROR]", f"Row {row_num}: BigQuery failed ({bq_err}) - rolling back Firestore", report)
                    try:
                        if target_snapshot is not None:
                            # target_ref.set(target_snapshot)
                            log("[ROLLBACK]", f"Row {row_num}: Firestore restored", report)
                        else:
                            target_ref.delete()
                            log("[ROLLBACK]", f"Row {row_num}: Firestore document deleted (new doc)", report)
                    except Exception as rb_err:
                        log("[CRITICAL]", f"Row {row_num}: rollback failed ({rb_err}); manual cleanup needed for {new_channel_id}", report)
                    continue

                status = "[DONE]" if bq_success else "[WARN]"
                log(status, f"Row {row_num}: mirrored {parent_id} -> {new_channel_id} ('{new_channel_name}', AI: {ai_status})", report)
                count += 1
    finally:
        if insights_client:
            insights_client.close()

    print("\n" + "=" * 60)
    print(f"  Mirror complete. Processed: {count} channel(s)")
    if dry_run:
        print("  Mode: DRY RUN - no writes were made")
    print("=" * 60)

    report_path = f"mirror_report_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.json"
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    print(f"  Report saved to: {report_path}\n")


In [8]:
# Configuration
csv_path = project_root / "src" / "dump" / "mirror.csv"
# csv_path = "E:\\Git\\lds\\backend\\practice\\py-script\\src\\dump\\mirror.csv"
dry_run = True
do_preview_ai = False

if do_preview_ai:
    sample_metadata = {
        "channel_id": "PARENT_CHANNEL_ID",
        "title": "Parent Channel",
        "description": "Parent channel description",
    }
    sample_new_name = "New Mirrored Channel"
    preview_context = f"{build_openai_context(sample_metadata, sample_new_name)}\n{SUMMARY_INSTRUCTIONS}"
    preview_client = ChannelInsightsClient(template=AI_TEMPLATE)
    preview_json = preview_client.describe_channel_json(sample_new_name, context=preview_context)
    print(preview_json[:500])
    preview_client.close()

if not csv_path.exists():
    raise FileNotFoundError(f"CSV not found: {csv_path}")

print("csv_path:", csv_path)
print("dry_run:", dry_run)
mirror_channels(str(csv_path), dry_run=dry_run)


csv_path: E:\Git\lds\backend\practice\py-script\src\dump\mirror.csv
dry_run: True


ValueError: Missing GCP_PROJECT_ID or FIRESTORE_DATABASE in env